# Sandbox — Silver Reviews

Exploração por trás de `silver_rules.tratar_reviews`.

> **Nota:** o problema do texto multilinha (comentários com quebras de linha e aspas) já foi resolvido na **Bronze** — ver `bronze/sandbox_landing_to_bronze`. Aqui os dados já chegam íntegros em Parquet, então o foco é só **tipagem**: `review_score` para int e as duas datas para timestamp.

In [ ]:
from utils.spark_utils import create_spark_session
from pyspark.sql.functions import col, trim, to_timestamp, current_timestamp

spark = create_spark_session('Sandbox-Silver-Reviews')

df_bronze = spark.read.parquet('s3a://bronze/olist/reviews')
df_bronze.show(5, truncate=True)
df_bronze.printSchema()

Conferindo a distribuição de `review_score` (deve ir de 1 a 5) — bom sinal de que a leitura multilinha na Bronze funcionou, pois valores fora dessa faixa indicariam linhas quebradas.

In [ ]:
df_bronze.groupBy('review_score').count().orderBy('review_score').show()

In [ ]:
df_silver = (df_bronze
    .withColumn('review_id', trim(col('review_id')))
    .withColumn('order_id', trim(col('order_id')))
    .withColumn('review_score', col('review_score').cast('int'))
    .withColumn('review_comment_title', trim(col('review_comment_title')))
    .withColumn('review_comment_message', trim(col('review_comment_message')))
    .withColumn('review_creation_date', to_timestamp(col('review_creation_date')))
    .withColumn('review_answer_timestamp', to_timestamp(col('review_answer_timestamp')))
    .withColumn('dt_criacao_silver', current_timestamp()))

df_silver.printSchema()
df_silver.show(5, truncate=True)

**Lógica promovida para `silver_rules.tratar_reviews`.**

In [ ]:
spark.stop()